- Carrega o ambiente, define o escopo temporal e recupera o calendário do staging.

In [2]:
import pandas as pd
import numpy as np

# configuração do escopo temporal
ANO_ATUAL = 2026
ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

# carrega a base de jogos salva na etapa de ingestão (00)
df_jogos = pd.read_parquet('./raw_data/raw_schedules.parquet')

# filtra apenas jogos que já foram concluídos (possuem placar definido)
df_jogos_validos = df_jogos[df_jogos['home_score'].notnull()].copy()

# ordenação cronológica estrita para garantir integridade temporal
df_jogos_validos = df_jogos_validos.sort_values(by=['season', 'week', 'gameday']).reset_index(drop=True)

print(f"Total de jogos processados: {len(df_jogos_validos)} partidas entre {min(ultimos_5_anos)} e {ANO_ATUAL}")

Total de jogos processados: 1424 partidas entre 2021 e 2026


- Para calcular métricas móveis sem duplicação de lógica e sem data leakage, desdobrar cada partida em duas linhas: uma sob a ótica do mandante e outra sob a ótica do visitante.

In [3]:
# perspectiva do mandante
df_home = df_jogos_validos[['game_id', 'season', 'week', 'gameday', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
df_home.columns = ['game_id', 'season', 'week', 'gameday', 'team', 'opponent', 'pf', 'pa']
df_home['is_home'] = 1

# perspectiva do visitante
df_away = df_jogos_validos[['game_id', 'season', 'week', 'gameday', 'away_team', 'home_team', 'away_score', 'home_score']].copy()
df_away.columns = ['game_id', 'season', 'week', 'gameday', 'team', 'opponent', 'pf', 'pa']
df_away['is_home'] = 0

# concatena e ordena por time e cronologia de jogo
df_team_games = pd.concat([df_home, df_away], ignore_index=True)
df_team_games = df_team_games.sort_values(by=['team', 'season', 'week', 'gameday']).reset_index(drop=True)

df_team_games.head(4)

,game_id,season,week,gameday,team,opponent,pf,pa,is_home
0,2021_01_ARI_TEN,2021,1,2021-09-12,ARI,TEN,38,13,0
1,2021_02_MIN_ARI,2021,2,2021-09-19,ARI,MIN,34,33,1
2,2021_03_ARI_JAX,2021,3,2021-09-26,ARI,JAX,31,19,0
3,2021_04_ARI_LA,2021,4,2021-10-03,ARI,LA,37,20,0


- Aplicando .shift(1) agrupado por time e temporada. Isso assegura que os pontos do jogo atual não entrem no cálculo da média pré-partida.

In [4]:
# fallback global: média geral da liga (utilizada apenas na Semana 1 do primeiro ano, 2021)
league_pf_mean = df_team_games['pf'].mean()
league_pa_mean = df_team_games['pa'].mean()

# fallback da franquia: média da temporada anterior do time (Season - 1)
team_season_means = df_team_games.groupby(['team', 'season'])[['pf', 'pa']].mean().reset_index()
# desloca a temporada em +1 para que o ano anterior case com a temporada atual no merge
team_season_means['season'] = team_season_means['season'] + 1
team_season_means = team_season_means.rename(columns={'pf': 'prior_pf', 'pa': 'prior_pa'})

# cálculo das médias móveis estritas dentro da temporada
grouped = df_team_games.groupby(['team', 'season'])

df_team_games['pf_last3'] = grouped['pf'].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
df_team_games['pa_last3'] = grouped['pa'].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
df_team_games['pf_season'] = grouped['pf'].transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
df_team_games['pa_season'] = grouped['pa'].transform(lambda x: x.shift(1).expanding(min_periods=1).mean())

# integração do prior da temporada anterior
df_team_games = df_team_games.merge(team_season_means, on=['team', 'season'], how='left')

# aplicação da cascata de tratamento de NaNs:
# se o cálculo atual for NaN -> Usa média do time no ano anterior -> Se ainda for NaN -> Usa média da liga
for metric in ['last3', 'season']:
    df_team_games[f'pf_{metric}'] = df_team_games[f'pf_{metric}'].fillna(df_team_games['prior_pf']).fillna(league_pf_mean)
    df_team_games[f'pa_{metric}'] = df_team_games[f'pa_{metric}'].fillna(df_team_games['prior_pa']).fillna(league_pa_mean)

# remove as colunas auxiliares temporárias
df_team_games = df_team_games.drop(columns=['prior_pf', 'prior_pa'])

# validação: verifica se restou algum NaN nas métricas calculadas
nans_restantes = df_team_games[['pf_last3', 'pa_last3', 'pf_season', 'pa_season']].isna().sum().to_dict()
print(f"Contagem de NaNs após tratamento: {nans_restantes}")

# exibe as primeiras linhas do primeiro time para validar a transição da Semana 1
df_team_games[['team', 'season', 'week', 'pf', 'pa', 'pf_last3', 'pa_last3', 'pf_season', 'pa_season']].head(5)

Contagem de NaNs após tratamento: {'pf_last3': 0, 'pa_last3': 0, 'pf_season': 0, 'pa_season': 0}


,team,season,week,pf,pa,pf_last3,pa_last3,pf_season,pa_season
0,ARI,2021,1,38,13,22.587781,22.587781,22.587781,22.587781
1,ARI,2021,2,34,33,38.000000,13.000000,38.000000,13.000000
2,ARI,2021,3,31,19,36.000000,23.000000,36.000000,23.000000
3,ARI,2021,4,37,20,34.333333,21.666667,34.333333,21.666667
4,ARI,2021,5,17,10,34.000000,24.000000,35.000000,21.250000


- Fundindo as features geradas de volta à tabela de jogos original (game_id), criando as colunas prefixadas home_ e away_.

In [5]:
# separa as estatísticas pré-jogo de mandante e visitante
stats_cols = ['game_id', 'team', 'pf_last3', 'pa_last3', 'pf_season', 'pa_season']

home_features = df_team_games[df_team_games['is_home'] == 1][stats_cols].copy()
home_features = home_features.rename(columns={
    'pf_last3': 'home_pf_last3',
    'pa_last3': 'home_pa_last3',
    'pf_season': 'home_pf_season',
    'pa_season': 'home_pa_season'
}).drop(columns=['team'])

away_features = df_team_games[df_team_games['is_home'] == 0][stats_cols].copy()
away_features = away_features.rename(columns={
    'pf_last3': 'away_pf_last3',
    'pa_last3': 'away_pa_last3',
    'pf_season': 'away_pf_season',
    'pa_season': 'away_pa_season'
}).drop(columns=['team'])

# consolidação da matriz base com o par PF e PA integrado
df_features_mestre = df_jogos_validos[['game_id', 'season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
df_features_mestre = df_features_mestre.merge(home_features, on='game_id', how='left')
df_features_mestre = df_features_mestre.merge(away_features, on='game_id', how='left')

# exibe o resultado final das variáveis de pontuação
df_features_mestre[['game_id', 'season', 'week', 'home_team', 'home_pf_last3', 'home_pa_last3', 'away_team', 'away_pf_last3', 'away_pa_last3']].tail(4)

,game_id,season,week,home_team,home_pf_last3,home_pa_last3,away_team,away_pf_last3,away_pa_last3
1420,2025_20_HOU_NE,2025,20,NE,32.0,7.666667,HOU,29.333333,17.333333
1421,2025_21_NE_DEN,2025,21,DEN,24.0,15.333333,NE,27.333333,9.666667
1422,2025_21_LA_SEA,2025,21,SEA,27.0,6.333333,LA,30.333333,22.666667
1423,2025_22_SEA_NE,2025,22,NE,18.0,8.666667,SEA,28.333333,12.000000


In [6]:
# carrega a base de estatísticas individuais
df_player_stats = pd.read_parquet('./raw_data/raw_player_stats.parquet')

# consolidação de turnovers ofensivos cometidos
df_player_stats['turnovers_committed'] = (
    df_player_stats['passing_interceptions'].fillna(0) +
    df_player_stats['rushing_fumbles_lost'].fillna(0) +
    df_player_stats['receiving_fumbles_lost'].fillna(0) +
    df_player_stats['sack_fumbles_lost'].fillna(0)
)

# consolidação de turnovers forçados pela defesa
df_player_stats['turnovers_forced'] = (
    df_player_stats['def_interceptions'].fillna(0) +
    df_player_stats['fumble_recovery_opp'].fillna(0)
)

# agregação de todas as métricas individuais por jogo e franquia
team_player_agg = df_player_stats.groupby(['game_id', 'team']).agg(
    off_pass_yds=('passing_yards', 'sum'),
    off_rush_yds=('rushing_yards', 'sum'),
    off_carries=('carries', 'sum'),
    off_turnovers=('turnovers_committed', 'sum'),
    def_sacks_produced=('def_sacks', 'sum'),
    def_turnovers_forced=('turnovers_forced', 'sum'),
    def_qb_hits=('def_qb_hits', 'sum')
).reset_index()

# limpeza de colunas existentes para garantir idempotência na execução
cols_player = [
    'off_pass_yds', 'off_rush_yds', 'off_carries', 
    'off_turnovers', 'def_sacks_produced', 'def_turnovers_forced', 'def_qb_hits'
]
cols_existentes = [c for c in cols_player if c in df_team_games.columns]
if cols_existentes:
    df_team_games = df_team_games.drop(columns=cols_existentes)

# integração na tabela time-jogo
df_team_games = df_team_games.merge(team_player_agg, on=['game_id', 'team'], how='left')
df_team_games[cols_player] = df_team_games[cols_player].fillna(0)

# validação das métricas agregadas do player stats
df_team_games[['team', 'season', 'week'] + cols_player].head(4)

,team,season,week,off_pass_yds,off_rush_yds,off_carries,off_turnovers,def_sacks_produced,def_turnovers_forced,def_qb_hits
0,ARI,2021,1,289,136,33,1,6.0,3,10
1,ARI,2021,2,400,103,21,2,1.0,0,4
2,ARI,2021,3,316,91,30,1,2.0,4,7
3,ARI,2021,4,268,216,40,0,0.0,2,5


In [7]:
# carrega a base de jogada a jogada
df_pbp = pd.read_parquet('./raw_data/raw_pbp.parquet')

# filtra jogadas regulares de scrimmage com dados de epa e sucesso válidos
filtro_base = (
    ((df_pbp['pass'] == 1) | (df_pbp['rush'] == 1)) &
    df_pbp['epa'].notnull() &
    df_pbp['success'].notnull()
)
df_pbp_valid = df_pbp[filtro_base]

# agregação de eficiência ofensiva agrupando pelo time com a posse
pbp_off = df_pbp_valid[df_pbp_valid['posteam'].notnull()].groupby(['game_id', 'posteam']).agg(
    off_epa_total=('epa', 'sum'),
    off_success_rate=('success', 'mean')
).reset_index().rename(columns={'posteam': 'team'})

# agregação de eficiência defensiva agrupando pelo time na defesa
pbp_def = df_pbp_valid[df_pbp_valid['defteam'].notnull()].groupby(['game_id', 'defteam']).agg(
    def_success_allowed=('success', 'mean')
).reset_index().rename(columns={'defteam': 'team'})

# agregação de faltas totais cometidas por franquia no jogo
filtro_penalties = (df_pbp['penalty'] == 1) & df_pbp['penalty_team'].notnull()
pbp_penalties = df_pbp[filtro_penalties].groupby(['game_id', 'penalty_team']).agg(
    team_penalty_yards=('penalty_yards', 'sum')
).reset_index().rename(columns={'penalty_team': 'team'})

# consolidação das visões de pbp
pbp_agg = pbp_off.merge(pbp_def, on=['game_id', 'team'], how='outer')
pbp_agg = pbp_agg.merge(pbp_penalties, on=['game_id', 'team'], how='outer')

# limpeza preventiva de colunas para permitir reexecuções limpas
cols_pbp = ['off_epa_total', 'off_success_rate', 'def_success_allowed', 'team_penalty_yards']
cols_existentes_pbp = [c for c in cols_pbp if c in df_team_games.columns]
if cols_existentes_pbp:
    df_team_games = df_team_games.drop(columns=cols_existentes_pbp)

# fusão com a tabela time-jogo
df_team_games = df_team_games.merge(pbp_agg, on=['game_id', 'team'], how='left')

# preenchimento de nans com valores neutros
df_team_games['off_epa_total'] = df_team_games['off_epa_total'].fillna(0)
df_team_games['off_success_rate'] = df_team_games['off_success_rate'].fillna(df_team_games['off_success_rate'].mean())
df_team_games['def_success_allowed'] = df_team_games['def_success_allowed'].fillna(df_team_games['def_success_allowed'].mean())
df_team_games['team_penalty_yards'] = df_team_games['team_penalty_yards'].fillna(0)

# validação das métricas do pbp integradas
df_team_games[['team', 'season', 'week'] + cols_pbp].head(4)

,team,season,week,off_epa_total,off_success_rate,def_success_allowed,team_penalty_yards
0,ARI,2021,1,13.562362,0.405797,0.430769,59.0
1,ARI,2021,2,6.324949,0.508197,0.508197,40.0
2,ARI,2021,3,11.343707,0.515152,0.522388,58.0
3,ARI,2021,4,21.334037,0.552632,0.597015,63.0


In [8]:
# lista completa de métricas de desempenho que receberão janelas móveis
features_para_processar = [
    'off_pass_yds', 'off_rush_yds', 'off_carries', 'off_turnovers',
    'def_sacks_produced', 'def_turnovers_forced', 'def_qb_hits',
    'off_epa_total', 'off_success_rate', 'def_success_allowed',
    'team_penalty_yards'
]

# fallback global da liga para partidas inaugurais sem histórico anterior
league_means = df_team_games[features_para_processar].mean()

# fallback da franquia baseado na temporada anterior
team_priors = df_team_games.groupby(['team', 'season'])[features_para_processar].mean().reset_index()
team_priors['season'] = team_priors['season'] + 1
prior_rename = {col: f'prior_{col}' for col in features_para_processar}
team_priors = team_priors.rename(columns=prior_rename)

# cálculo das médias móveis estritas dentro do mesmo ano
grouped = df_team_games.groupby(['team', 'season'])
for col in features_para_processar:
    df_team_games[f'{col}_last3'] = grouped[col].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    df_team_games[f'{col}_season'] = grouped[col].transform(lambda x: x.shift(1).expanding(min_periods=1).mean())

# fusão com os priors da temporada passada
cols_prior_existentes = [c for c in prior_rename.values() if c in df_team_games.columns]
if cols_prior_existentes:
    df_team_games = df_team_games.drop(columns=cols_prior_existentes)
df_team_games = df_team_games.merge(team_priors, on=['team', 'season'], how='left')

# aplicação da cascata de imputação para preencher nans em lote
for col in features_para_processar:
    for metric in ['last3', 'season']:
        target_col = f'{col}_{metric}'
        prior_col = f'prior_{col}'
        df_team_games[target_col] = df_team_games[target_col].fillna(df_team_games[prior_col]).fillna(league_means[col])

# limpeza das colunas temporárias de priors
df_team_games = df_team_games.drop(columns=list(prior_rename.values()))

# validação de integridade estrutural
check_cols = [f'{c}_{m}' for c in features_para_processar for m in ['last3', 'season']]
print("nans restantes após processamento unificado:", df_team_games[check_cols].isna().sum().sum())

nans restantes após processamento unificado: 0


In [9]:
# mapeamento de todas as colunas de janelas móveis geradas
cols_to_merge = [f'{c}_{m}' for c in features_para_processar for m in ['last3', 'season']]
cols_integration = ['game_id', 'team'] + cols_to_merge

# preparação das colunas do time mandante
home_stats = df_team_games[df_team_games['is_home'] == 1][cols_integration].copy()
home_rename = {col: f'home_{col}' for col in cols_to_merge}
home_stats = home_stats.rename(columns=home_rename).drop(columns=['team'])

# preparação das colunas do time visitante
away_stats = df_team_games[df_team_games['is_home'] == 0][cols_integration].copy()
away_rename = {col: f'away_{col}' for col in cols_to_merge}
away_stats = away_stats.rename(columns=away_rename).drop(columns=['team'])

# remoção preventiva de colunas antigas na tabela mestra para permitir reexecução
cols_existentes_mestre = [c for c in list(home_rename.values()) + list(away_rename.values()) if c in df_features_mestre.columns]
if cols_existentes_mestre:
    df_features_mestre = df_features_mestre.drop(columns=cols_existentes_mestre)

# fusão na matriz mestra de confrontos
df_features_mestre = df_features_mestre.merge(home_stats, on='game_id', how='left')
df_features_mestre = df_features_mestre.merge(away_stats, on='game_id', how='left')

# exibe dimensões e amostra final da tabela mestra consolidada
print(f"formato da tabela mestra: {df_features_mestre.shape}")
df_features_mestre[['game_id', 'home_team', 'away_team', 'home_off_epa_total_last3', 'away_off_epa_total_last3']].tail(4)

formato da tabela mestra: (1424, 59)


,game_id,home_team,away_team,home_off_epa_total_last3,away_off_epa_total_last3
1420,2025_20_HOU_NE,NE,HOU,16.598049,-0.102406
1421,2025_21_NE_DEN,DEN,NE,0.399789,0.844308
1422,2025_21_LA_SEA,SEA,LA,5.576549,9.021680
1423,2025_22_SEA_NE,NE,SEA,-8.118595,10.812090


In [10]:
# converte a data do jogo para formato datetime para permitir operacao temporal
df_team_games['gameday'] = pd.to_datetime(df_team_games['gameday'])

# ordena cronologicamente por franquia e temporada
df_team_games = df_team_games.sort_values(by=['team', 'season', 'gameday']).reset_index(drop=True)

# calcula a diferenca em dias desde a ultima partida disputada pelo time no mesmo ano
df_team_games['days_since_last_game'] = (
    df_team_games.groupby(['team', 'season'])['gameday'].diff().dt.days
)

# marca flag de descanso prolongado para intervalos iguais ou superiores a 13 dias
# na semana inaugural da temporada o valor ausente e tratado como sem descanso extra
df_team_games['has_bye'] = (df_team_games['days_since_last_game'] >= 13).astype(int)

# separa a informacao de bye para mandante e visitante
bye_home = df_team_games[df_team_games['is_home'] == 1][['game_id', 'has_bye']].rename(
    columns={'has_bye': 'home_has_bye'}
)
bye_away = df_team_games[df_team_games['is_home'] == 0][['game_id', 'has_bye']].rename(
    columns={'has_bye': 'away_has_bye'}
)

# extrai o indicador nativo de jogo divisional da base de jogos
div_info = df_jogos_validos[['game_id', 'div_game']].copy()
div_info['div_game'] = div_info['div_game'].fillna(0).astype(int)

# consolida as informacoes contextuais em um dataframe intermediario
context_features = div_info.merge(bye_home, on='game_id', how='left')
context_features = context_features.merge(bye_away, on='game_id', how='left')

# preenche eventuais lacunas e calcula o saldo liquido de vantagem de descanso
context_features['home_has_bye'] = context_features['home_has_bye'].fillna(0).astype(int)
context_features['away_has_bye'] = context_features['away_has_bye'].fillna(0).astype(int)
context_features['vantagem_descanso'] = context_features['home_has_bye'] - context_features['away_has_bye']

# limpa eventuais colunas contextuais pre existentes na tabela mestra para permitir reexecucao
cols_contexto = ['div_game', 'home_has_bye', 'away_has_bye', 'vantagem_descanso']
cols_existentes_contexto = [c for c in cols_contexto if c in df_features_mestre.columns]
if cols_existentes_contexto:
    df_features_mestre = df_features_mestre.drop(columns=cols_existentes_contexto)

# funde as variaveis contextuais diretamente na matriz mestra de confrontos
df_features_mestre = df_features_mestre.merge(context_features, on='game_id', how='left')

# exibe distribuicao do confronto divisional e da vantagem de descanso
print("frequencia de partidas divisionais:")
print(df_features_mestre['div_game'].value_counts())
print("\nfrequencia do saldo de vantagem de descanso (-1: visitante com bye, 0: neutro, 1: mandante com bye):")
print(df_features_mestre['vantagem_descanso'].value_counts())

# inspeciona os dados finais integrados na matriz mestra
df_features_mestre[[
    'game_id', 'home_team', 'away_team', 
    'div_game', 'home_has_bye', 'away_has_bye', 'vantagem_descanso'
]].tail(5)

frequencia de partidas divisionais:
div_game
0    932
1    492
Name: count, dtype: int64

frequencia do saldo de vantagem de descanso (-1: visitante com bye, 0: neutro, 1: mandante com bye):
vantagem_descanso
 0    1283
 1      82
-1      59
Name: count, dtype: int64


,game_id,home_team,away_team,div_game,home_has_bye,away_has_bye,vantagem_descanso
1419,2025_20_LA_CHI,CHI,LA,0,0,0,0
1420,2025_20_HOU_NE,NE,HOU,0,0,0,0
1421,2025_21_NE_DEN,DEN,NE,0,0,0,0
1422,2025_21_LA_SEA,SEA,LA,1,0,0,0
1423,2025_22_SEA_NE,NE,SEA,0,1,1,0


In [11]:
# carrega a base de lesoes gerada na etapa de ingestao
df_injuries = pd.read_parquet('./raw_data/raw_injuries.parquet')

# identifica colunas de tentativas de passe e identificador de jogador no player stats
col_att = 'attempts' if 'attempts' in df_player_stats.columns else 'passing_attempts'
col_id = 'player_id' if 'player_id' in df_player_stats.columns else ('gsis_id' if 'gsis_id' in df_player_stats.columns else 'player_name')

# identifica o qb titular absoluto da franquia no ano baseado no volume total de tentativas
qb_passers = df_player_stats[df_player_stats[col_att] > 0]
qb_season_tot = qb_passers.groupby(['season', 'team', col_id])[col_att].sum().reset_index()
qb_season_tot = qb_season_tot.sort_values(by=['season', 'team', col_att], ascending=[True, True, False])
qb1_franchise = qb_season_tot.groupby(['season', 'team']).first().reset_index()
qb1_franchise = qb1_franchise[['season', 'team', col_id]].rename(columns={col_id: 'qb1_id'})

# identifica o passador principal de cada partida individual
game_passers = qb_passers.groupby(['game_id', 'team', col_id])[col_att].sum().reset_index()
game_passers = game_passers.sort_values(by=['game_id', 'team', col_att], ascending=[True, True, False])
game_starter_qb = game_passers.groupby(['game_id', 'team']).first().reset_index()
game_starter_qb = game_starter_qb[['game_id', 'team', col_id]].rename(columns={col_id: 'game_qb_id'})

# cruza o titular do jogo com o titular do ano para acionar a flag de reserva
game_starter_qb = game_starter_qb.merge(
    df_team_games[['game_id', 'team', 'season']].drop_duplicates(), 
    on=['game_id', 'team'], 
    how='left'
)
game_starter_qb = game_starter_qb.merge(qb1_franchise, on=['season', 'team'], how='left')
game_starter_qb['is_backup_qb'] = (game_starter_qb['game_qb_id'] != game_starter_qb['qb1_id']).astype(int)

# filtra desfalques de linha ofensiva listados como fora ou duvidosos
posicoes_ol = ['C', 'G', 'OG', 'OT', 'T', 'OL', 'C/G', 'G/C', 'T/G', 'G/T']
filtro_ol = (
    df_injuries['position'].isin(posicoes_ol) &
    df_injuries['report_status'].isin(['Out', 'Doubtful'])
)
df_ol_out = df_injuries[filtro_ol]

# conta o total de baixas na linha ofensiva por time, semana e temporada
ol_counts = df_ol_out.groupby(['season', 'week', 'team']).size().reset_index(name='o_line_out')

# limpa colunas pre existentes para garantir idempotencia
cols_instabilidade = ['is_backup_qb', 'o_line_out']
cols_existentes_inst = [c for c in cols_instabilidade if c in df_team_games.columns]
if cols_existentes_inst:
    df_team_games = df_team_games.drop(columns=cols_existentes_inst)

# integra as variaveis de instabilidade na tabela time jogo
df_team_games = df_team_games.merge(
    game_starter_qb[['game_id', 'team', 'is_backup_qb']], 
    on=['game_id', 'team'], 
    how='left'
)
df_team_games = df_team_games.merge(
    ol_counts, 
    on=['season', 'week', 'team'], 
    how='left'
)

# preenche ausencias com zero para manter integridade numerica
df_team_games['is_backup_qb'] = df_team_games['is_backup_qb'].fillna(0).astype(int)
df_team_games['o_line_out'] = df_team_games['o_line_out'].fillna(0).astype(int)

# visualizacao das metricas brutas agregadas
df_team_games[['team', 'season', 'week', 'is_backup_qb', 'o_line_out']].head(5)

,team,season,week,is_backup_qb,o_line_out
0,ARI,2021,1,0,0
1,ARI,2021,2,0,0
2,ARI,2021,3,0,2
3,ARI,2021,4,0,0
4,ARI,2021,5,0,1


In [12]:
# prepara visao do mandante para as variaveis de instabilidade
home_inst = df_team_games[df_team_games['is_home'] == 1][['game_id', 'is_backup_qb', 'o_line_out']].copy()
home_inst = home_inst.rename(columns={
    'is_backup_qb': 'home_is_backup_qb',
    'o_line_out': 'home_o_line_out'
})

# prepara visao do visitante para as variaveis de instabilidade
away_inst = df_team_games[df_team_games['is_home'] == 0][['game_id', 'is_backup_qb', 'o_line_out']].copy()
away_inst = away_inst.rename(columns={
    'is_backup_qb': 'away_is_backup_qb',
    'o_line_out': 'away_o_line_out'
})

# limpa colunas antigas na tabela mestra para evitar duplicatas em reexecucoes
cols_final_merge = ['home_is_backup_qb', 'home_o_line_out', 'away_is_backup_qb', 'away_o_line_out']
cols_existentes_mestre = [c for c in cols_final_merge if c in df_features_mestre.columns]
if cols_existentes_mestre:
    df_features_mestre = df_features_mestre.drop(columns=cols_existentes_mestre)

# funde as variaveis de instabilidade na tabela mestra
df_features_mestre = df_features_mestre.merge(home_inst, on='game_id', how='left')
df_features_mestre = df_features_mestre.merge(away_inst, on='game_id', how='left')

# preenche lacunas eventuais com zero
df_features_mestre[cols_final_merge] = df_features_mestre[cols_final_merge].fillna(0).astype(int)

# checagem global de nans em todas as colunas do dataset
total_nans = df_features_mestre.isna().sum().sum()
print(f"status do dataset consolidado: {df_features_mestre.shape[0]} partidas x {df_features_mestre.shape[1]} colunas")
print(f"total de nans restantes em toda a tabela mestra: {total_nans}")

# visualizacao final das 4 variaveis de contexto e instabilidade
colunas_exibicao = [
    'game_id', 'home_team', 'away_team', 
    'home_is_backup_qb', 'away_is_backup_qb',
    'home_o_line_out', 'away_o_line_out',
    'vantagem_descanso', 'div_game'
]
df_features_mestre[colunas_exibicao].tail(5)

status do dataset consolidado: 1424 partidas x 67 colunas
total de nans restantes em toda a tabela mestra: 0


,game_id,home_team,away_team,home_is_backup_qb,away_is_backup_qb,home_o_line_out,away_o_line_out,vantagem_descanso,div_game
1419,2025_20_LA_CHI,CHI,LA,0,0,0,0,0,0
1420,2025_20_HOU_NE,NE,HOU,0,0,0,0,0,0
1421,2025_21_NE_DEN,DEN,NE,1,0,0,0,0,0
1422,2025_21_LA_SEA,SEA,LA,0,0,1,1,0,1
1423,2025_22_SEA_NE,NE,SEA,0,0,0,0,0,0


In [14]:
display(df_features_mestre.columns)

Index(['game_id', 'season', 'week', 'home_team', 'away_team', 'home_score',
       'away_score', 'home_pf_last3', 'home_pa_last3', 'home_pf_season',
       'home_pa_season', 'away_pf_last3', 'away_pa_last3', 'away_pf_season',
       'away_pa_season', 'home_off_pass_yds_last3', 'home_off_pass_yds_season',
       'home_off_rush_yds_last3', 'home_off_rush_yds_season',
       'home_off_carries_last3', 'home_off_carries_season',
       'home_off_turnovers_last3', 'home_off_turnovers_season',
       'home_def_sacks_produced_last3', 'home_def_sacks_produced_season',
       'home_def_turnovers_forced_last3', 'home_def_turnovers_forced_season',
       'home_def_qb_hits_last3', 'home_def_qb_hits_season',
       'home_off_epa_total_last3', 'home_off_epa_total_season',
       'home_off_success_rate_last3', 'home_off_success_rate_season',
       'home_def_success_allowed_last3', 'home_def_success_allowed_season',
       'home_team_penalty_yards_last3', 'home_team_penalty_yards_season',
       'away

In [15]:
# lista base de metricas de desempenho para calculo de diferencas liquidas
metricas_desempenho = [
    'pf', 'pa', 'off_pass_yds', 'off_rush_yds', 'off_carries', 'off_turnovers',
    'def_sacks_produced', 'def_turnovers_forced', 'def_qb_hits',
    'off_epa_total', 'off_success_rate', 'def_success_allowed', 'team_penalty_yards'
]

# calcula a diferenca liquida mandante menos visitante para cada metrica e janela
diff_cols = []
for col in metricas_desempenho:
    for janela in ['last3', 'season']:
        col_home = f'home_{col}_{janela}'
        col_away = f'away_{col}_{janela}'
        diff_col_name = f'diff_{col}_{janela}'
        
        df_features_mestre[diff_col_name] = df_features_mestre[col_home] - df_features_mestre[col_away]
        diff_cols.append(diff_col_name)

# diferenca liquida para as variaveis de instabilidade
df_features_mestre['diff_is_backup_qb'] = df_features_mestre['home_is_backup_qb'] - df_features_mestre['away_is_backup_qb']
df_features_mestre['diff_o_line_out'] = df_features_mestre['home_o_line_out'] - df_features_mestre['away_o_line_out']
diff_cols.extend(['diff_is_backup_qb', 'diff_o_line_out'])

# definicao do alvo de classificacao binaria para probabilidade de vitoria
df_features_mestre['target_home_win'] = (df_features_mestre['home_score'] > df_features_mestre['away_score']).astype(int)

In [17]:
# exportacao do dataset consolidado para formato parquet
caminho_exportacao = './processed_data/df_features_modelagem.parquet'
df_features_mestre.to_parquet(caminho_exportacao, index=False)

print("processamento concluido com sucesso")
print(f"dataset salvo em: {caminho_exportacao}")
print(f"dimensoes finais: {df_features_mestre.shape[0]} partidas x {df_features_mestre.shape[1]} colunas")
print(f"total de diferenciais liquidos gerados: {len(diff_cols)}")

# inspeciona amostra dos diferenciais liquidos e alvo binario
cols_inspecao = [
    'game_id', 'home_team', 'away_team',
    'diff_off_epa_total_last3', 'diff_off_success_rate_last3',
    'diff_o_line_out', 'vantagem_descanso', 'target_home_win'
]
df_features_mestre[cols_inspecao].tail(5)

processamento concluido com sucesso
dataset salvo em: ./processed_data/df_features_modelagem.parquet
dimensoes finais: 1424 partidas x 96 colunas
total de diferenciais liquidos gerados: 28


,game_id,home_team,away_team,diff_off_epa_total_last3,diff_off_success_rate_last3,diff_o_line_out,vantagem_descanso,target_home_win
1419,2025_20_LA_CHI,CHI,LA,4.822347,-0.038171,0,0,0
1420,2025_20_HOU_NE,NE,HOU,16.700455,0.089149,0,0,1
1421,2025_21_NE_DEN,DEN,NE,-0.444519,-0.029738,0,0,0
1422,2025_21_LA_SEA,SEA,LA,-3.445130,0.023573,0,0,1
1423,2025_22_SEA_NE,NE,SEA,-18.930684,-0.145783,0,0,0
